# PhoBERT -- Baseline (no augmentation)\n\nTrain trên `aug_baseline.csv` (86,719 dòng, không augmentation), đánh giá trên `dev.csv`/`test.csv` đã đóng băng (Giai đoạn 1). Đây là điểm neo (anchor) để so sánh với 4 thí nghiệm augmentation.\n\n**Chạy trên Google Colab** -- cần mount Drive và có sẵn secret `HF_TOKEN` (biểu tượng chìa khoá bên trái > Add new secret).

## Dependencies

In [1]:
!pip install -q transformers datasets huggingface_hub scikit-learn accelerate sentencepiece py_vncorenlp
!pip install -q -U datasets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 14.5 MB/s eta 0:00:00


In [2]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: Enable GPU in Runtime > Change runtime type.")

Device: cuda
GPU: Tesla T4


In [3]:
from google.colab import userdata
from huggingface_hub import login, whoami, HfApi

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
user_info = whoami()
print(f"Authenticated as: {user_info['name']}")

Authenticated as: AnoraLee


In [4]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Hate_Speech_Detection")
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
AUGMENTED_DIR = DATA_DIR / "augmented"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_DIR}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/Hate_Speech_Detection


In [5]:
MODEL_NAME = "vinai/phobert-base"
MAX_LENGTH = 128
SEED = 42
LABELS = ["CLEAN", "OFFENSIVE", "HATE"]

label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

## Chuẩn hoá & Phân từ dùng chung

In [6]:
import os
import re
import unicodedata
import pandas as pd
import py_vncorenlp

def text_key(text):
    """Normalize a text for exact-duplicate / leakage matching."""
    text = unicodedata.normalize("NFKC", str(text)).strip().lower()
    return re.sub(r"\s+", " ", text)

def add_key(df, source_col="text_raw"):
    df = df.copy()
    df["_key"] = df[source_col].map(text_key)
    return df

_vncorenlp_dir = "/content/vncorenlp"
if not os.path.exists(_vncorenlp_dir):
    os.makedirs(_vncorenlp_dir, exist_ok=True)
    py_vncorenlp.download_model(save_dir=_vncorenlp_dir)
rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=_vncorenlp_dir)

def segment_text(text):
    try:
        sentences = rdrsegmenter.word_segment(str(text))
        return " ".join(sentences)
    except Exception:
        return str(text)

def apply_segmentation(df, source_col="text_raw", target_col="text"):
    df = df.copy()
    df[target_col] = df[source_col].map(segment_text)
    return df

In [7]:
TOXIC_TEENCODE_MAP = {
    r"\bko\b": "không", r"\bhok\b": "không", r"\bdc\b": "được", r"\bđc\b": "được",
    r"\bj\b": "gì", r"\bbt\b": "bình thường", r"\btrc\b": "trước", r"\bnhg\b": "nhưng",
    r"\bthg\b": "thằng",
    r"\bdm\b": "địt mẹ", r"\bđm\b": "địt mẹ", r"\bdkm\b": "địt con mẹ", r"\bđkm\b": "địt con mẹ",
    r"\bvkl\b": "vãi lồn", r"\bvcl\b": "vãi lồn", r"\bvl\b": "vãi lồn", r"\bkl\b": "cái lồn",
    r"\bcc\b": "cục cứt", r"\bcđm\b": "cộng đồng mạng", r"\bml\b": "mặt lồn",
    r"\bđjt\b": "địt", r"\bdjt\b": "địt", r"\bdit\b": "địt",
    r"\bloz\b": "lồn", r"\blon\b": "lồn",
    r"\bcac\b": "cặc", r"\bcặk\b": "cặc", r"\bđb\b": "đầu buồi",
    r"\bcút\b": "cút", r"\bđĩ\b": "đĩ", r"\bphò\b": "phò"
}

def normalize_teencode(text):
    for pattern, replacement in TOXIC_TEENCODE_MAP.items():
        text = re.sub(pattern, replacement, str(text), flags=re.IGNORECASE)
    return text

def apply_teencode_normalization(df, col="text_raw"):
    df = df.copy()
    df[col] = df[col].apply(normalize_teencode)
    return df

## Load dev/test đã đóng băng (Giai đoạn 1)

In [8]:
dev_raw = pd.read_csv(PROCESSED_DIR / "dev.csv")
test_raw = pd.read_csv(PROCESSED_DIR / "test.csv")

dev_raw = dev_raw.rename(columns={"text": "text_raw"}) if "text_raw" not in dev_raw.columns else dev_raw
test_raw = test_raw.rename(columns={"text": "text_raw"}) if "text_raw" not in test_raw.columns else test_raw

dev_df = apply_teencode_normalization(dev_raw.dropna(subset=["text_raw"]).reset_index(drop=True))
test_df = apply_teencode_normalization(test_raw.dropna(subset=["text_raw"]).reset_index(drop=True))

dev_keys = set(dev_df["text_raw"].map(text_key))
test_keys = set(test_df["text_raw"].map(text_key))

dev_df = apply_segmentation(dev_df)
test_df = apply_segmentation(test_df)

print(f"Validation: {dev_df.shape}")
print(f"Test: {test_df.shape}")

Validation: (2650, 5)
Test: (6576, 5)


## Load & làm sạch train (aug_baseline.csv)

In [9]:
train_raw = pd.read_csv(AUGMENTED_DIR / "aug_baseline.csv")
train_raw = apply_teencode_normalization(train_raw)
train_raw = add_key(train_raw)

conflict_keys = set(
    train_raw.groupby("_key")["label"].nunique().loc[lambda c: c > 1].index
)
blocked_keys = conflict_keys | dev_keys | test_keys

train_df = (
    train_raw[~train_raw["_key"].isin(blocked_keys)]
    .drop_duplicates("_key")
    .drop(columns="_key")
    .reset_index(drop=True)
)
print(f"Train: {len(train_raw):,} -> {len(train_df):,} dòng "
      f"(loại {len(conflict_keys)} nhãn xung đột + leakage với dev/test)")
print(train_df["label"].value_counts().to_dict())

Train: 85,127 -> 84,275 dòng (loại 12 nhãn xung đột + leakage với dev/test)
{'CLEAN': 59276, 'OFFENSIVE': 19421, 'HATE': 5578}


## Tokenizer + hàm dùng chung

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def prepare_split(df):
    output = df[["text", "label"]].dropna().copy()
    assert output["label"].isin(LABELS).all(), "Unexpected label found."
    output["label"] = output["label"].map(label2id)
    return Dataset.from_pandas(output, preserve_index=False)

def build_tokenized_dataset(train_df, dev_df, test_df):
    dataset = DatasetDict({
        "train": prepare_split(train_df),
        "validation": prepare_split(dev_df),
        "test": prepare_split(test_df),
    })
    tokenized = dataset.map(tokenize_function, batched=True)
    tokenized = tokenized.rename_column("label", "labels")
    model_columns = [
        c for c in ["input_ids", "attention_mask", "token_type_ids", "labels"]
        if c in tokenized["train"].column_names
    ]
    tokenized.set_format("torch", columns=model_columns)
    print(tokenized)
    return tokenized

In [11]:
train_df = apply_segmentation(train_df)
tokenized = build_tokenized_dataset(train_df, dev_df, test_df)

Map:   0%|          | 0/84275 [00:00<?, ? examples/s]

Map:   0%|          | 0/2650 [00:00<?, ? examples/s]

Map:   0%|          | 0/6576 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 84275
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2650
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 6576
    })
})


## Metrics

In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    """Unified metric schema -- MUST stay identical across all 3 notebooks
    so results in the report are directly comparable."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
        "hate_f1": f1_score(labels, predictions, labels=[LABELS.index("HATE")], average="macro"),
    }

## Train

In [15]:
import json
from transformers import TrainingArguments, AutoModelForSequenceClassification, Trainer, set_seed, EarlyStoppingCallback

set_seed(SEED)

def train_experiment(experiment_name, model_dir_name, tokenized, push_to_hub_repo=None):
    """Load a FRESH PhoBERT model and train one experiment end to end.
    Always instantiates a new model -- never reuses a model/trainer object
    from a previous cell, to avoid accidental weight leakage between runs."""
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, label2id=label2id, id2label=id2label,
    )
    model_dir = MODELS_DIR / model_dir_name

    training_args = TrainingArguments(
        output_dir=str(model_dir / "_checkpoints"),
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=10,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        report_to="none",
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()

    test_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
    print(f"[{experiment_name}] test metrics: {test_metrics}")

    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)

    experiment_results_dir = RESULTS_DIR / experiment_name
    experiment_results_dir.mkdir(parents=True, exist_ok=True)

    metrics_path = experiment_results_dir / f"metrics_{experiment_name}.json"
    with open(metrics_path, "w") as f:
        json.dump(test_metrics, f, indent=2)

    print(f"Model saved to: {model_dir}")
    print(f"Metrics saved to: {metrics_path}")

    if push_to_hub_repo:
        api = HfApi()
        api.create_repo(repo_id=push_to_hub_repo, repo_type="model", private=True, exist_ok=True)
        trainer.model.push_to_hub(push_to_hub_repo, private=True,
                                  commit_message=f"Upload PhoBERT {experiment_name} model")
        tokenizer.push_to_hub(push_to_hub_repo, private=True,
                              commit_message="Upload PhoBERT tokenizer")
        api.upload_file(
            path_or_fileobj=str(metrics_path),
            path_in_repo="test_metrics.json",
            repo_id=push_to_hub_repo, repo_type="model",
        )
        print(f"Uploaded: https://huggingface.co/{push_to_hub_repo}")

    return trainer, test_metrics

In [16]:
trainer, test_metrics = train_experiment(
    experiment_name="baseline",
    model_dir_name="baseline_phobert",
    tokenized=tokenized,
    push_to_hub_repo=f"{whoami()['name']}/vietnamese-hsd-phobert-baseline",
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Hate F1
1,0.414132,0.444777,0.832075,0.595998,0.832142,0.500000
2,0.380821,0.445882,0.835472,0.603992,0.837407,0.526767
3,0.280307,0.567224,0.797358,0.590254,0.816130,0.527835


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...8dep_y8/model.safetensors:   0%|          |  549kB /  540MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Uploaded: https://huggingface.co/AnoraLee/vietnamese-hsd-phobert-baseline
